
CineMatch — TMDB BGE-M3 Embeddings + FAISS Index Builder
=========================================================
Encodes the full TMDB catalog (~1.37M movies) using BAAI/bge-m3 (1024-dim)
and builds an exact-search FAISS IndexIDMap2(FlatIP) for cosine retrieval.

In [ ]:
#! pip install sentence-transformers faiss-gpu-cu12 recbole requests python-dotenv

In [2]:
from __future__ import annotations

import gc
import json
import os
import sys
import time
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
from multiprocessing import Pool

import math
from transformers import AutoTokenizer
from tqdm import tqdm
from FlagEmbedding import BGEM3FlagModel

In [ ]:
# CONFIG  

MODEL_ID = "BAAI/bge-m3"

# Encoding
ENCODE_OUTER_BATCH = 4096   # rows per checkpoint cycle
ENCODE_INNER_BATCH = 512  # rows per model.encode() call
INDEX_ADD_BATCH    = 100_000  # rows per faiss.add_with_ids()

# Validation
FORCE_REENCODE          = False
VALIDATION_SAMPLE_SIZE  = 2000
MIN_NORM_MEAN           = 0.70
MAX_ZERO_NORM_FRAC      = 0.05
VALIDATE_MIN_WRITTEN    = 4096

# Sanity check
SANITY_TOP_K = 10

# PATHS  
def detect_paths() -> dict:

    # Colab
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        base = Path("/content/drive/MyDrive/cinematch")
        print("Runtime: Colab")
    except ImportError:
        # HPC
        hpc = Path("/blue/egn6933/nagabhairava.r")
        if hpc.exists():
            base = hpc
            print("Runtime: HPC")
        else:
            # Local
            here = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()
            for candidate in [here, *here.parents]:
                if (candidate / "Data").exists() and (candidate / "src").exists():
                    base = candidate / "Data"
                    break
            else:
                base = Path.cwd() / "Data"
            print("Runtime: Local")

    out = base / "outputs" / "tmdbbgefaiss"
    out.mkdir(parents=True, exist_ok=True)

    catalog_path = base / "tmdb_semantic_catalog_alllangs_with_new_movies.csv"

    return {
        "base":       base,
        "catalog":    catalog_path,
        "emb_mmap":   out / "tmdb_bge_m3_embeddings.float32.mmap",
        "checkpoint": out / "tmdb_bge_m3_checkpoint.json",
        "faiss":      out / "tmdb_bge_m3_flatip.faiss",
        "manifest":   out / "tmdb_bge_m3_build_manifest.json",
        "out_dir":    out,
    }


In [ ]:
# HELPERS  

def sample_norm_stats(
    arr: np.memmap, row_count: int,
    sample_size: int = VALIDATION_SAMPLE_SIZE, seed: int = 7
) -> tuple[float, float, float, float]:

    if row_count <= 0:
        return 0.0, 0.0, 0.0, 1.0
    take = min(sample_size, row_count)
    rng = np.random.default_rng(seed)
    idx = rng.choice(row_count, size=take, replace=False)
    sample = np.asarray(arr[idx], dtype="float32")
    norms = np.linalg.norm(sample, axis=1)
    zero_frac = float((norms == 0).mean())
    return float(norms.min()), float(norms.mean()), float(norms.max()), zero_frac

In [ ]:
def main():
    t_start = time.time()
    paths = detect_paths()

    # Load catalog
    print(f"\n{'─'*60}")
    print(f"  Loading TMDB catalog: {paths['catalog']}")
    print(f"{'─'*60}")

    assert paths["catalog"].exists(), f"Missing catalog CSV: {paths['catalog']}"

    df = pd.read_csv(paths["catalog"], low_memory=False)
    df["id"] = pd.to_numeric(df["id"], errors="coerce")
    df = df.dropna(subset=["id"]).copy()
    df["id"] = df["id"].astype(int)
    df["movieDoc"] = df["movieDoc"].fillna("").astype(str)

    mask = df["movieDoc"].str.contains("Plot:", na=False) & (df["movieDoc"].str.len() >= 80)
    df_embed = df.loc[mask, ["id", "movieDoc"]].copy()

    ids   = df_embed["id"].to_numpy(dtype=np.int64)
    texts = df_embed["movieDoc"].tolist()
    order = np.argsort([len(t) for t in texts])
    texts = [texts[i] for i in order]
    ids   = ids[order]
    n_rows = len(texts)

    print(f"  Total catalog: {len(df):,}")
    print(f"  Embeddable rows (has Plot + ≥80 chars): {n_rows:,}")

    if "original_language" in df.columns:
        lang_counts = df.loc[mask, "original_language"].value_counts()
        print(f"\n  Language distribution (embeddable rows):")
        for lang in ["en", "te", "hi", "ja", "ko", "ta", "ml", "kn"]:
            print(f"    {lang}: {lang_counts.get(lang, 0):,}")

    # Load model
    print(f"\n{'─'*60}")
    print(f"  Loading {MODEL_ID}")

    if torch.cuda.is_available():
        device = "cuda"
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = "mps"
    else:
        device = "cpu"

    model = BGEM3FlagModel(MODEL_ID, use_fp16=True, device=device)
    emb_dim = 1024

    print(f"  Device:  {device}")
    print(f"  Model:   {MODEL_ID}")
    print(f"  Emb dim: {emb_dim}")

    # torch.compile
    model.model = torch.compile(model.model, mode="reduce-overhead")
    model.model.eval()
    with torch.no_grad():
        _ = model.encode(
            ["warmup"],
            return_dense=True,
            return_sparse=False,
            return_colbert_vecs=False,
            max_length=64,
        )

    # Memmap setup
    print(f"\n{'─'*60}")
    print(f"  Encoding {n_rows:,} rows to {paths['emb_mmap']}")

    expected_bytes = n_rows * emb_dim * np.dtype("float32").itemsize

    ckpt_state = {}
    if paths["checkpoint"].exists():
        ckpt_state = json.loads(paths["checkpoint"].read_text(encoding="utf-8"))

    reset_reasons = []
    if FORCE_REENCODE:
        reset_reasons.append("FORCE_REENCODE=True")
    if paths["emb_mmap"].exists() and paths["emb_mmap"].stat().st_size != expected_bytes:
        reset_reasons.append(
            f"mmap size mismatch: expected={expected_bytes}, actual={paths['emb_mmap'].stat().st_size}"
        )
    if ckpt_state:
        if ckpt_state.get("model_id") and ckpt_state["model_id"] != MODEL_ID:
            reset_reasons.append(f"model mismatch: {ckpt_state['model_id']} != {MODEL_ID}")
        if int(ckpt_state.get("embedding_dim", -1)) not in (-1, emb_dim):
            reset_reasons.append(f"dim mismatch: {ckpt_state['embedding_dim']} != {emb_dim}")

    need_reset = len(reset_reasons) > 0
    if need_reset:
        print("  Resetting due to:")
        for r in reset_reasons:
            print(f"    - {r}")
        for f in [paths["emb_mmap"], paths["checkpoint"], paths["faiss"]]:
            if f.exists():
                f.unlink()

    start_idx = 0
    if not need_reset and ckpt_state:
        start_idx = min(max(int(ckpt_state.get("next_index", 0)), 0), n_rows)

    mmap_mode = "r+" if paths["emb_mmap"].exists() else "w+"
    emb_memmap = np.memmap(
        paths["emb_mmap"], mode=mmap_mode, dtype="float32", shape=(n_rows, emb_dim)
    )

    if start_idx >= VALIDATE_MIN_WRITTEN and not need_reset:
        nmin, nmean, nmax, zfrac = sample_norm_stats(emb_memmap, start_idx)
        print(f"  Checkpoint prefix norms @ {start_idx:,}: "
              f"min={nmin:.4f} mean={nmean:.4f} max={nmax:.4f} zero_frac={zfrac:.4f}")
        if nmean < MIN_NORM_MEAN or zfrac > MAX_ZERO_NORM_FRAC:
            print("  Corrupted prefix — restarting from scratch.")
            for f in [paths["emb_mmap"], paths["checkpoint"], paths["faiss"]]:
                if f.exists():
                    f.unlink()
            emb_memmap = np.memmap(
                paths["emb_mmap"], mode="w+", dtype="float32", shape=(n_rows, emb_dim)
            )
            start_idx = 0

    if start_idx >= n_rows:
        print(f"  Encoding already complete ({n_rows:,} rows).\n")
    else:
        print(f"  Starting from index {start_idx:,}")

    # Encode
    embed_t0 = time.time()

    for start in range(start_idx, n_rows, ENCODE_OUTER_BATCH):
        end = min(start + ENCODE_OUTER_BATCH, n_rows)
        batch_texts = texts[start:end]

        with torch.no_grad():
            vectors = model.encode(
                batch_texts,
                batch_size=ENCODE_INNER_BATCH,
                return_dense=True,
                return_sparse=False,
                return_colbert_vecs=False,
                max_length=1024,
            )["dense_vecs"].astype("float32")

        emb_memmap[start:end] = vectors

        if (end % (ENCODE_OUTER_BATCH * 10) == 0) or (end == n_rows):
            emb_memmap.flush()
            state = {
                "model_id": MODEL_ID,
                "embedding_dim": int(emb_dim),
                "next_index": int(end),
                "completed": bool(end == n_rows),
                "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
            }
            paths["checkpoint"].write_text(json.dumps(state, indent=2), encoding="utf-8")
            print(f"  Encoded {end:,}/{n_rows:,}")

        del vectors
        gc.collect()

    emb_memmap.flush()
    encode_seconds = time.time() - embed_t0
    print(f"  Encoding complete in {encode_seconds:.1f}s")

    # Validate norms
    nmin, nmean, nmax, zfrac = sample_norm_stats(emb_memmap, n_rows)
    print(f"\n  Final norm stats: min={nmin:.4f} mean={nmean:.4f} max={nmax:.4f} zero_frac={zfrac:.4f}")
    assert nmean >= MIN_NORM_MEAN, f"Norm mean too low: {nmean}"
    assert zfrac <= MAX_ZERO_NORM_FRAC, f"Too many zero norms: {zfrac}"

    # Build FAISS index
    print(f"\n{'─'*60}")
    print(f"  Building FAISS IndexIDMap2(FlatIP) with {n_rows:,} vectors")
    print(f"{'─'*60}")

    faiss_t0 = time.time()
    base_index = faiss.IndexFlatIP(emb_dim)
    index = faiss.IndexIDMap2(base_index)

    for start in range(0, n_rows, INDEX_ADD_BATCH):
        end = min(start + INDEX_ADD_BATCH, n_rows)
        batch_vecs = np.array(emb_memmap[start:end])
        batch_ids  = ids[start:end]
        index.add_with_ids(batch_vecs, batch_ids)
        print(f"  Added rows {start:,} to {end:,}")

    faiss.write_index(index, str(paths["faiss"]))
    faiss_seconds = time.time() - faiss_t0
    print(f"  Saved: {index.ntotal:,} vectors → {paths['faiss']}")
    print(f"  FAISS build time: {faiss_seconds:.1f}s")

    # Build manifest
    manifest = {
        "catalog": {
            "source": str(paths["catalog"]),
            "total_rows": int(len(df)),
            "embedded_rows": int(n_rows),
        },
        "embedding": {
            "model_id": MODEL_ID,
            "embedding_dim": int(emb_dim),
            "device": device,
            "rows_encoded": int(n_rows),
            "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
            "norm_check": {
                "sample_min": nmin,
                "sample_mean": nmean,
                "sample_max": nmax,
                "sample_zero_frac": zfrac,
            },
            "encoding": {
                "outer_batch": ENCODE_OUTER_BATCH,
                "inner_batch": ENCODE_INNER_BATCH,
                "duration_seconds": round(encode_seconds, 2),
                "memmap_path": str(paths["emb_mmap"]),
                "checkpoint_path": str(paths["checkpoint"]),
            },
            "faiss": {
                "index_type": "IndexIDMap2(IndexFlatIP)",
                "ntotal": int(index.ntotal),
                "dimension": int(emb_dim),
                "add_batch": INDEX_ADD_BATCH,
                "duration_seconds": round(faiss_seconds, 2),
                "faiss_path": str(paths["faiss"]),
            },
        },
    }
    paths["manifest"].write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"\n  Manifest saved: {paths['manifest']}")

    # Sanity check
    print(f"\n  Top-{SANITY_TOP_K} retrieval sanity check")

    meta_cols = ["id", "title", "original_language", "genres"]
    meta_cols = [c for c in meta_cols if c in df.columns]
    meta = df[meta_cols].copy()
    meta["id"] = meta["id"].astype(np.int64)
    meta = meta.drop_duplicates("id").set_index("id")

    queries = [
        "emotional family drama with sacrifice and redemption",
        "Telugu romantic comedy village",
        "Japanese anime sci-fi mecha",
    ]
    for q in queries:
        qvec = model.encode(
            [q],
            return_dense=True,
            return_sparse=False,
            return_colbert_vecs=False,
        )["dense_vecs"].astype("float32")
        D, I = index.search(qvec, SANITY_TOP_K)
        print(f"\n  Query: \"{q}\"")
        for rank, (mid, score) in enumerate(zip(I[0], D[0]), 1):
            if mid < 0:
                continue
            row = meta.loc[mid] if mid in meta.index else None
            title = row["title"] if row is not None else f"id={mid}"
            lang  = row.get("original_language", "?") if row is not None else "?"
            print(f"    {rank:2}. [{lang}] {title}  (score={score:.4f})")

    total_time = time.time() - t_start
    print(f"\n{'═'*60}")
    print(f"  DONE — Total time: {total_time:.1f}s")
    print(f"{'═'*60}")


if __name__ == "__main__":
    main()
    

Runtime: HPC

────────────────────────────────────────────────────────────
  Loading TMDB catalog: /blue/egn6933/nagabhairava.r/tmdb_semantic_catalog_alllangs_with_new_movies.csv
────────────────────────────────────────────────────────────
  Total catalog: 1,367,793
  Embeddable rows (has Plot + ≥80 chars): 1,366,255

  Language distribution (embeddable rows):
    en: 745,174
    te: 3,216
    hi: 8,920
    ja: 60,300
    ko: 14,995
    ta: 5,257
    ml: 4,449
    kn: 1,723

────────────────────────────────────────────────────────────
  Loading BAAI/bge-m3


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

long.jpg:   0%|          | 0.00/485k [00:00<?, ?B/s]

.DS_Store:   0%|          | 0.00/6.15k [00:00<?, ?B/s]

nqa.jpg:   0%|          | 0.00/158k [00:00<?, ?B/s]

...

  Encoded 1,351,680/1,366,255


Inference Embeddings: 100%|██████████| 5/5 [00:01<00:00,  3.43it/s]
/scratch/local/27652863/ipykernel_2520765/4287785602.py:153: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  "timestamp_utc": pd.Timestamp.utcnow().isoformat(),


  Encoded 1,366,255/1,366,255
  Encoding complete in 420.4s

  Final norm stats: min=0.9999 mean=1.0000 max=1.0001 zero_frac=0.0000

────────────────────────────────────────────────────────────
  Building FAISS IndexIDMap2(FlatIP) with 1,366,255 vectors
────────────────────────────────────────────────────────────
  Added rows 0 to 100,000
  Added rows 100,000 to 200,000
  Added rows 200,000 to 300,000
  Added rows 300,000 to 400,000
  Added rows 400,000 to 500,000
  Added rows 500,000 to 600,000
  Added rows 600,000 to 700,000
  Added rows 700,000 to 800,000
  Added rows 800,000 to 900,000
  Added rows 900,000 to 1,000,000
  Added rows 1,000,000 to 1,100,000
  Added rows 1,100,000 to 1,200,000
  Added rows 1,200,000 to 1,300,000
  Added rows 1,300,000 to 1,366,255
  Saved: 1,366,255 vectors → /blue/egn6933/nagabhairava.r/outputs/tmdbbgefaiss/tmdb_bge_m3_flatip.faiss
  FAISS build time: 10.8s

  Manifest saved: /blue/egn6933/nagabhairava.r/outputs/tmdbbgefaiss/tmdb_bge_m3_build_manifest

/scratch/local/27652863/ipykernel_2520765/4287785602.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  "timestamp_utc": pd.Timestamp.utcnow().isoformat(),



  Query: "emotional family drama with sacrifice and redemption"
     1. [en] The Cross  (score=0.6285)
     2. [en] Redemption  (score=0.6264)
     3. [ku] Sacrifice  (score=0.6241)
     4. [fr] Redemption  (score=0.6229)
     5. [en] Redemption for Easter  (score=0.6179)
     6. [en] The Redemption  (score=0.6164)
     7. [en] An Avonlea Christmas  (score=0.6143)
     8. [en] Redemption  (score=0.6137)
     9. [en] Torch of Remembrance  (score=0.6127)
    10. [sl] Family Therapy  (score=0.6121)

  Query: "Telugu romantic comedy village"
     1. [te] Kundanapu Bomma  (score=0.6201)
     2. [ml] Village Guys  (score=0.6173)
     3. [te] Rarandoi Veduka Chudham  (score=0.6131)
     4. [ta] Balle Vellaiyathevaa  (score=0.6056)
     5. [te] Missamma  (score=0.6020)
     6. [ta] Desingu Raja  (score=0.5968)
     7. [ta] Munthiri Kaadu  (score=0.5950)
     8. [bn] Sutorang  (score=0.5943)
     9. [hu] A Village Romance  (score=0.5914)
    10. [en] Villagelo Vinayakudu  (score=0.5896)

  Que